# UDTF Registration and Basic Queries

This notebook demonstrates how to register UDTFs in a Spark session and query them using SQL.

## Prerequisites

- Spark cluster with active Spark session
- CDF credentials in `config.toml` file
- Generated UDTF files (see `basic_generation.ipynb`)


## Step 1: Install Dependencies


In [ ]:
# Install required packages
# Note: Restart Python kernel after installation when prompted

%pip install cognite-pygen-spark cognite-sdk pyspark


## Step 2: Load Configuration and Setup


In [ ]:
from cognite.pygen import load_cognite_client_from_toml
import tomli
from pathlib import Path
import sys

# Load credentials from TOML file
with open("config.toml", "rb") as f:
    config = tomli.load(f)

cognite_config = config["cognite"]

# Add generated code directory to Python path
udtf_dir = Path("./generated_udtfs")
sys.path.insert(0, str(udtf_dir))

print("✓ Configuration loaded")
print(f"  Project: {cognite_config['project']}")
print(f"  Cluster: {cognite_config['cdf_cluster']}")


## Step 3: Register UDTFs


In [ ]:
from pyspark.sql.functions import udtf

# Import the generated UDTF class
# Replace 'SmallBoatUDTF' with your actual generated class name
from cognite_udtfs.SmallBoat_udtf import SmallBoatUDTF

# Wrap the class with udtf()
smallboat_udtf = udtf(SmallBoatUDTF)

# Register in Spark session
spark.udtf.register("smallboat_udtf", smallboat_udtf)

print("✓ UDTF registered: smallboat_udtf")


## Step 4: Query UDTF with Credentials from Config


In [ ]:
# Query UDTF using credentials from config file
query = f"""
SELECT * FROM smallboat_udtf(
    client_id => '{cognite_config["client_id"]}',
    client_secret => '{cognite_config["client_secret"]}',
    tenant_id => '{cognite_config["tenant_id"]}',
    cdf_cluster => '{cognite_config["cdf_cluster"]}',
    project => '{cognite_config["project"]}',
    name => NULL,
    description => NULL
) LIMIT 10;
"""

# Execute query
result = spark.sql(query)
result.show(truncate=False)


## Step 5: Query with Filters


In [ ]:
# Query with filter parameter
query = f"""
SELECT * FROM smallboat_udtf(
    client_id => '{cognite_config["client_id"]}',
    client_secret => '{cognite_config["client_secret"]}',
    tenant_id => '{cognite_config["tenant_id"]}',
    cdf_cluster => '{cognite_config["cdf_cluster"]}',
    project => '{cognite_config["project"]}',
    name => 'MyBoat',
    description => NULL
) LIMIT 10;
"""

result = spark.sql(query)
result.show(truncate=False)
